# SAC debug run on Colab — v2.4.2

**Purpose:** smoke-test the v2.4.2 fixes (`target_entropy = -13` and WandB integration) with a short 50k-step run on Colab's free T4 GPU. Expected runtime ~12 minutes. **Not for production training** — Colab free tier disconnects too aggressively for the full 2M-step run; use Kaggle for that.

**Before running:**
1. `Runtime → Change runtime type → Hardware accelerator → T4 GPU`. Confirm by checking the resource indicator (top right).
2. Add your WandB API key as a Colab Secret:
   - Left sidebar → key icon (Secrets) → `+ Add new secret`.
   - Name: `WANDB_API_KEY`. Value: paste the key from https://wandb.ai/settings.
   - Toggle "Notebook access" ON.
3. Optional but recommended: mount Google Drive (Cell 1 does this automatically) so results survive even if Colab disconnects.

**What to look for** on the WandB dashboard while it runs:
- `train/ent_coef` should decline gradually, NOT collapse below 0.01 by step 25k.
- `rollout/ep_len_mean` should climb toward 93 (full season).
- `rollout/ep_rew_mean` should not get catastrophically worse over time.
- System tab: GPU utilization >50%, RAM stable.

If `ent_coef` collapses anyway → kill the run, switch to fixed `ent_coef = 0.1` (uncomment the `hp_overrides` line in Cell 4).

In [ ]:
# CELL 1: Mount Drive, clone repo, install deps
import subprocess, sys, os

# Mount Google Drive so results survive Colab disconnects
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_MOUNTED = True
    DRIVE_ROOT = '/content/drive/MyDrive/thesis_results'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f'\u2713  Drive mounted. Results will be copied to: {DRIVE_ROOT}')
except Exception as e:
    DRIVE_MOUNTED = False
    print(f'\u26a0  Drive mount failed ({type(e).__name__}: {e}).')
    print('   Results will only be saved to /content/ (lost on disconnect).')

# Clone latest repo
if os.path.exists('/content/thesis'):
    subprocess.run(['rm', '-rf', '/content/thesis'], check=True)
subprocess.run([
    'git', 'clone',
    'https://github.com/taratorbati/thesis.git',
    '/content/thesis'
], check=True)

os.chdir('/content/thesis')
sys.path.insert(0, '/content/thesis')

# Install deps
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'stable-baselines3==2.6.0',
    'gymnasium==1.1.1',
    'wandb>=0.16',
    'pytest',
], check=True)

import numpy as np, gymnasium, stable_baselines3 as sb3, torch
print(f'numpy:             {np.__version__}')
print(f'gymnasium:         {gymnasium.__version__}')
print(f'stable-baselines3: {sb3.__version__}')
print(f'PyTorch:           {torch.__version__}')
print(f'CUDA:              {torch.cuda.is_available()}')
try:
    import wandb
    print(f'wandb:             {wandb.__version__}')
except ImportError:
    print('wandb:             NOT INSTALLED')
torch.set_num_threads(4)
print('Setup complete.')

In [ ]:
# CELL 2: Load WandB API key + verify GPU
import os
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
    os.environ['WANDB_API_KEY'] = key
    print('\u2713  WANDB_API_KEY loaded from Colab Secrets.')
except Exception as e:
    print(f'\u26a0  Could not load WANDB_API_KEY from Colab Secrets ({type(e).__name__}).')
    print('   Training will run with local-only logging.')
    print('   To enable: left sidebar \u2192 key icon \u2192 add WANDB_API_KEY.')

# Confirm GPU is allocated
print()
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'nvidia-smi failed — no GPU?')

In [ ]:
# CELL 3: Smoke tests + benchmark (~2 min)
# Runs the pytest suite to catch any import-time errors before training,
# then a quick step-rate benchmark.
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v'],
    capture_output=False,
)
assert result.returncode == 0, 'Smoke tests failed — do not proceed to training.'

import time
from stable_baselines3 import SAC
from src.rl.gym_env import IrrigationEnv
from src.rl.networks import CTDESACPolicy, make_sac_policy_kwargs
from src.rl.train import train_sac  # catches train.py import errors

env = IrrigationEnv(randomize=False)
obs, _ = env.reset()
model = SAC(
    policy=CTDESACPolicy, env=env,
    policy_kwargs=make_sac_policy_kwargs(N=130),
    buffer_size=10_000, batch_size=256,
    learning_starts=10, gradient_steps=1, verbose=0,
)
model.learn(total_timesteps=50, reset_num_timesteps=True)
t0 = time.time()
model.learn(total_timesteps=200, reset_num_timesteps=False)
rate = 200 / (time.time() - t0)
print(f'\nDevice: {model.device}')
print(f'Rate:   {rate:.1f} steps/sec  ->  est {50_000/rate/60:.1f} min for 50k steps')
print('PASSED' if obs.shape[0] == 707 else 'FAILED: wrong obs_dim')

In [ ]:
# CELL 4: Debug training (50k steps, ~12 min)
from src.rl.train import train_sac

SEED = 0

# If ent_coef still collapses with target_entropy=-13, uncomment the next line
# to fall back to a fixed entropy coefficient.
# hp_overrides = {'ent_coef': 0.1}
hp_overrides = None

train_sac(
    total_timesteps=50_000,            # debug-run size; NOT for thesis
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    checkpoint_freq=10_000,            # smaller for short run
    eval_freq=5_000,
    hp_overrides=hp_overrides,
    wandb_project='sac-irrigation-thesis',
    verbose=1,
)

In [ ]:
# CELL 5: Copy results to Google Drive (so they survive Colab disconnects)
import shutil, os, datetime

if not DRIVE_MOUNTED:
    print('Drive was not mounted; results are only in /content/ and will be lost on disconnect.')
else:
    src = '/content/thesis/results/rl'
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    dst = f'{DRIVE_ROOT}/colab_debug_runs/run_{timestamp}'

    # Skip replay buffer (multi-GB and not useful for a debug run)
    def ignore_replay_buffers(directory, files):
        return [f for f in files if '_replay_buffer' in f or 'replay_buffer_latest' in f]

    shutil.copytree(src, dst, ignore=ignore_replay_buffers)

    total_mb = 0
    file_count = 0
    for root, dirs, files in os.walk(dst):
        for f in files:
            total_mb += os.path.getsize(os.path.join(root, f)) / 1e6
            file_count += 1

    print(f'\u2713  Copied {file_count} files ({total_mb:.1f} MB) to:')
    print(f'   {dst}')
    print()
    print('Files preserved (no replay buffers):')
    for root, dirs, files in os.walk(dst):
        for f in files:
            rel = os.path.relpath(os.path.join(root, f), dst)
            mb = os.path.getsize(os.path.join(root, f)) / 1e6
            print(f'  {rel}  ({mb:.2f} MB)')